# Lesson 29 Lab — Custom Kernels: Packing, Dequantization, and CUTLASS Boundaries

**Puzzle:** When is an INT4 pack/dequant kernel worth building instead of using an existing backend?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

An INT4 execution path contains pack/storage, scale loads, unpack/dequant, GEMM, epilogue, launches, and integration with framework layouts and streams.

### Core mechanism

The end-to-end budget is `T = T_pack/load + T_dequant + T_gemm + T_epilogue + overhead`. Fusing stages can remove intermediate traffic; a composed PyTorch reference intentionally exposes that unfused cost.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "29-custom-int4-kernels"
device = require_cuda()
torch.manual_seed(2026 + 29)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

A specialized CUTLASS/Triton kernel may win on stable shapes but costs engineering, testing, portability, and maintenance. Mature libraries remain the baseline to beat.

### What this code tests

The lab validates nibble semantics and times a composed unpack-dequant-matmul reference, labeling it explicitly as non-fused and non-CUTLASS.

**Experiment:** Validate vectorized INT4 nibble packing/unpacking and time the composed PyTorch dequantize-plus-matmul path against BF16.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
m,k,n=32,4096,4096; x=torch.randn(m,k,device=device,dtype=torch.bfloat16); w=torch.randn(n,k,device=device,dtype=torch.bfloat16); q,scales,_=symmetric_quantize(w,bits=4,group_size=128)
codes=(q.to(torch.int16)&0xF).flatten(); packed=(codes[0::2]|(codes[1::2]<<4)).to(torch.uint8)
def composed():
    lo=(packed.to(torch.int16)&15); hi=((packed.to(torch.int16)>>4)&15); u=torch.stack([lo,hi],1).flatten(); u=torch.where(u>=8,u-16,u).reshape(n,k).float()
    dq=(u.reshape(n,k//128,128)*scales[...,None]).reshape(n,k).bfloat16(); return x@dq.t()
packed_t=cuda_benchmark(composed,warmup=3,repeats=10); bf16_t=cuda_benchmark(lambda:x@w.t(),warmup=5,repeats=15)
result=base_result(29,"pytorch-gpu"); result.update({"shape_mkn":[m,k,n],"packed_bytes":packed.numel(),"bf16_timing":bf16_t,"composed_unpack_dequant_matmul":packed_t,
    "implementation":"composed PyTorch reference, not fused CUTLASS","conclusion":"The composed reference exposed integration overhead; it is a semantic baseline, not a custom-kernel performance claim."})


## 3. Inspect the evidence

Use the result to locate overhead, not to claim CUTLASS performance. A custom-kernel project begins only after a measured gap and stable shapes.

### Acceptance and rollback gate

First locate a repeated shape-level gap, verify pack/dequant semantics, profile roofline and memory traffic, implement, then require end-to-end gain and quality across the target shape distribution.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "bf16_timing": {
    "median_ms": 0.027136,
    "p90_ms": 0.027488,
    "repeats": 15,
    "samples_ms": [
      0.027136,
      0.028192,
      0.027424,
      0.027456,
      0.027136,
      0.027488,
      0.027136,
      0.02704,
      0.027072,
      0.026848,
      0.02704,
      0.026912,
      0.026112,
      0.027904,
      0.026432
    ],
    "warmup": 5
  },
  "composed_unpack_dequant_matmul": {
    "median_ms": 0.32872,
    "p90_ms": 0.330656,
    "repeats": 10,
    "samples_ms": [
      0.330656,
      0.330304,
      0.32256,
      0.330848,
      0.325088,
      0.32976,
      0.322944,
      0.330496,
      0.322944,
      0.32768
    ],
    "warmup": 3
  },
  "conclusion": "The composed reference exposed integration overhead; it is a semantic baseline, not a custom-kernel performance claim.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
 

## 4. Explain the result

Build custom code when the existing backend misses an important, repeated shape and the recoverable end-to-end budget exceeds integration cost.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).